In [ ]:
import zipfile
from os import listdir, remove, makedirs, rename, getenv
from os.path import exists, sep
from re import compile, match, findall
import pandas as pd
import numpy as np
SEPARATOR = sep
EU4_DIR = getenv("EU4_INSTALL_LOCATION")
TAGS_FILE = EU4_DIR + SEPARATOR + "common" + SEPARATOR + "country_tags" + SEPARATOR + "00_countries.txt"
DESIRABLE_PROVINCE_DATA = ["num_of_times_developed_var", "owner", "controller", ]

df_provinces = pd.read_csv("./data/provinces_stage2.csv")

In [2]:
class ProvinceDevelopment():
    def __init__(self, tax = 0, production = 0, manpower = 0):
        self.tax = tax
        self.production = production
        self.manpower = manpower
    def get_total_development(self):
        return self.tax + self.production + self.manpower
class ProvinceContext():
    def __init__(self, identifier = None, development = None, in_buildings_section = False, in_history_section = False, changed_owner_num = 0, changed_controller_num = 0, num_buildings = 0, current_owner = None, development_clicks = 0):
        self.identifier = identifier
        self.development = ProvinceDevelopment() if development is None else development
        self.in_history_section = in_history_section
        self.in_buildings_section = in_buildings_section
        self.changed_owner_num = changed_owner_num
        self.changed_controller_num = changed_controller_num
        self.num_buildings = num_buildings
        self.current_owner = current_owner
        self.development_clicks = development_clicks
    def get_old_development(self):
        row = df_provinces.loc[df_provinces["province_id"] == self.identifier]
        if row.empty:
            self.development = ProvinceDevelopment(0, 0, 0)
        else:
            assert len(row) == 1, f"Expected one source row for province {self.identifier}, found {len(row)}"
            source = row.iloc[0]
            self.development = ProvinceDevelopment(int(source.base_tax), int(source.base_production), int(source.base_manpower))
        return
    def store(self, save_context):
        if self.development.tax == 0 or self.development.production == 0 or self.development.manpower == 0:
            self.get_old_development()
        save_context.provinces.append(self)
        return save_context
class SaveFileGamestateContext():
    def __init__(self, prev_line = '', in_players_countries = False, in_province = None, player_countries = None, players = None, provinces = None, done = False):
        self.prev_line = prev_line
        self.in_players_countries = in_players_countries
        self.in_province = in_province
        self.player_countries = [] if player_countries is None else player_countries
        self.players = [] if players is None else players
        self.provinces = [] if provinces is None else provinces
        self.done = done
    def finalise(self):
        assert(self.done)
        self.players_countries_list = list(zip(self.players, self.player_countries))
        player_by_country = dict(zip(self.player_countries, self.players))
        list_of_provinces_dicts = []
        for province in self.provinces:
            list_of_provinces_dicts.append({"id": province.identifier, "new_tax": province.development.tax, "new_production": province.development.production, "new_manpower": province.development.manpower,
                                            "changed_owner_num": province.changed_owner_num, "changed_controller_num": province.changed_controller_num, "num_buildings": province.num_buildings,
                                            "current_owner": province.current_owner, "development_clicks": province.development_clicks,
                                            "is_player_owned": province.current_owner in player_by_country, "player": player_by_country.get(province.current_owner)})
        self.provinces = pd.DataFrame(list_of_provinces_dicts)
        self.provinces["save"] = self.file_name
        self.provinces["is_test_save"] = self.file_name.startswith("Test " )
        return

In [3]:
def extract_country_tags():
    tags = []
    with open(TAGS_FILE) as file:
        for line in file.readlines():
            line = line.strip()
            if line.startswith("#") or len(line) < 3:
                continue
            else:
                tags.append(line[0:3])    
    return tags

In [4]:
def extract_eu4_save(save_path, output_dir):
    # Ensure the output directory exists
    makedirs(output_dir, exist_ok=True)
    
    try:
        # Open the .eu4 file as a standard zip archive
        with zipfile.ZipFile(save_path, 'r') as zip_ref:
            # Extract all contents (meta, gamestate, ai), but remove the ai file afterwards since it is redundant
            zip_ref.extractall(output_dir)
            split_save_path = save_path.split('/')
            original_save_file_name = split_save_path[len(split_save_path) - 1]
            for file in zip_ref.namelist():
                if file != 'ai':
                    rename(output_dir + SEPARATOR + file, output_dir + SEPARATOR + original_save_file_name + '-' + file)
        remove(output_dir + SEPARATOR + "ai")
    except zipfile.BadZipFile:
        print("Error: This file is not a compressed zip archive.")
        print("It might be uncompressed plaintext or a binary Ironman save.")
    return

# Example usage
save_files_path = "./data/save_files"
save_files = listdir(save_files_path)
output_folder = "./data/parsed_save_files"
for file in listdir(output_folder):
    remove(output_folder + SEPARATOR + file)
for save_file in save_files:
    extract_eu4_save(save_files_path + SEPARATOR + save_file, output_folder)


In [5]:
tags = extract_country_tags()

Now for every save file, I'm going to extract all province data

In [6]:
PLAYER_FLAG = "players_countries={"
PROVINCE_FLAG = compile(r'(-\d{1,4}={)')
COUNTRY_FLAG = "countries={"
CLOSE_SCOPE = "}"
OWNER_CHANGE_REGEX = compile(r'^owner="(\w{3})"$')
DEVELOPMENT_CLICKS_KEYS = {"num_of_times_developed_var", "num_of_times_developed"}
CONTROLLER_OPEN_SCOPE_REGEX = r'controller={'
TAG_ASSIGNMENT_REGEX = compile(r'(tag="\w{3}")')
YEAR_SCOPE = compile(r'(\d{4}.\d{1,2}.\d{1,2}={)')
BUILDINGS = ["courthouse", "town_hall", "university", "workshop", "counting_house", "temple", "cathedral", "marketplace", "trade_depot", "stock_exchange", "dock", "drydock", "shipyard", "grand_shipyard", "coastal_defence", "naval_battery", "barracks", "training_fields", "regimental_camp", "conscription_center", "fort_15th", "fort_16th", "fort_17th", "fort_18th", "farm_estate", "weapons", "textile", "plantations", "tradecompany", "mills", "wharf", "furnace", "state_house", "naval_equipment_manufactory"]
HISTORY_SECTION = r'history={'
BUILDINGS_SECTION = r'buildings={'

def identify_line(line, context):
    if line is None or line == '' or line.startswith('#'):
        context.prev_line = line
        return context # Empty line or commented out
    if line == PLAYER_FLAG:
        context.in_players_countries = True
        context.prev_line = line
        return context
    if line == CLOSE_SCOPE:
        if context.in_players_countries:
            context.in_players_countries = False
            context.prev_line = line
            return context
    if context.in_players_countries:
        if line.startswith('"') and line.endswith('"'):
            tag_or_player = line.strip('"')
            if tag_or_player in tags:
                tag = tag_or_player
                context.player_countries.append(tag)
            else: 
                player = tag_or_player
                context.players.append(player)
        context.prev_line = line
        return context
    if match(PROVINCE_FLAG, line):
        if context.in_province is not None:
            context.in_province.store(context)
        identifier = int(line.lstrip("-").rstrip("={"))
        province = ProvinceContext(identifier)
        context.in_province = province
        context.prev_line = line
        return context
    if context.in_province is not None:
        if line == CLOSE_SCOPE and context.in_province.in_buildings_section:
            context.in_province.in_buildings_section = False
            context.prev_line = line
            return context
        if context.in_province.in_buildings_section:
            if '=' in line:
                key, value = line.split('=', 1)
                if key in BUILDINGS and value == 'yes':
                    context.in_province.num_buildings += 1
            context.prev_line = line
            return context
        if '=' in line:
            key, value = line.split('=', 1)
            if key in DEVELOPMENT_CLICKS_KEYS:
                context.in_province.development_clicks = int(float(value))
                context.prev_line = line
                return context
        owner_match = match(OWNER_CHANGE_REGEX, line)
        if owner_match:
            owner_tag = owner_match.group(1)
            if not context.in_province.in_history_section and context.in_province.current_owner is None:
                context.in_province.current_owner = owner_tag
            else:
                context.in_province.changed_owner_num += 1
            context.prev_line = line
            return context
        if line == CONTROLLER_OPEN_SCOPE_REGEX:
            context.prev_line = line
            return context
        if context.prev_line == CONTROLLER_OPEN_SCOPE_REGEX and match(TAG_ASSIGNMENT_REGEX, line):
            context.in_province.changed_controller_num += 1
            return context
        if line == HISTORY_SECTION:
            context.in_province.in_history_section = True
            context.prev_line = line
            return context
        if context.in_province.in_history_section == False:
            if '=' in line:
                parts = line.split("=")
                key = parts[0]
                value = parts[1]
                if line.startswith("base"):
                    if key == 'base_tax' and context.in_province.development.tax == 0:
                        context.in_province.development.tax = int(float(value))
                    if key == 'base_production' and context.in_province.development.production == 0:
                        context.in_province.development.production = int(float(value))
                    if key == 'base_manpower' and context.in_province.development.manpower == 0:
                        context.in_province.development.manpower = int(float(value))
                if line == BUILDINGS_SECTION:
                    context.in_province.in_buildings_section = True
                context.prev_line = line
                return context
    if line == COUNTRY_FLAG:
        if context.in_province is not None:
            context.in_province.store(context)
        context.done = True
        context.prev_line = line
        return context
    context.prev_line = line
    return context

In [7]:
PARSED_SAVE_FOLDER = output_folder
saves = []
for save_file in listdir(PARSED_SAVE_FOLDER):
    file_path = PARSED_SAVE_FOLDER + SEPARATOR + save_file
    with open(file_path) as file:
        if save_file.endswith('gamestate'):
            context = SaveFileGamestateContext()
            context.file_name = save_file.removesuffix(".eu4-gamestate")
            for line in file.readlines():
                line = line.strip()
                if context.done:
                    context.finalise()
                    break
                context = identify_line(line, context)
            saves.append(context)
        elif save_file.endswith('meta'):
            continue
all_saves_provinces = pd.DataFrame()
for save in saves:
    all_saves_provinces = pd.concat([all_saves_provinces, save.provinces], ignore_index=True)

In [8]:
all_saves_provinces = all_saves_provinces.rename(columns={"id": "province_id"})

In [9]:
source_provinces = df_provinces.drop(
    columns="Unnamed: 0",
    errors="ignore",
).copy()

all_saves_provinces["province_id"] = (
    all_saves_provinces["province_id"].astype(int)
)
source_provinces["province_id"] = source_provinces["province_id"].astype(int)

joined_dfs = all_saves_provinces.merge(
    source_provinces,
    how="left",
    left_on="province_id",
    right_on="province_id",
    suffixes=("_save", "_original"),
    validate="many_to_one",
)

In [10]:
LEGACY_VALUE_DEFINERS = ["changed_tax", "changed_production", "changed_manpower", "changed_owner_num", "num_buildings"]
TARGET_COLUMN = "investment_preference_target"

In [11]:
joined_dfs["changed_tax"] = joined_dfs["new_tax"] - joined_dfs["base_tax"].fillna(0).astype(int)
joined_dfs["changed_production"] = joined_dfs["new_production"] - joined_dfs["base_production"].fillna(0).astype(int)
joined_dfs["changed_manpower"] = joined_dfs["new_manpower"] - joined_dfs["base_manpower"].fillna(0).astype(int)
joined_dfs["legacy_heuristic_score"] = joined_dfs[LEGACY_VALUE_DEFINERS].sum(axis=1)

# The behavioral target is defined only where a real save province is currently
# owned by a human player and has a matching source-data row. Test saves are
# retained for parser checks but excluded from model labels.
joined_dfs["has_source_data"] = joined_dfs["name"].notna()
joined_dfs["target_eligible"] = (
    joined_dfs["is_player_owned"]
    & ~joined_dfs["is_test_save"]
    & joined_dfs["has_source_data"]
)
joined_dfs[TARGET_COLUMN] = np.nan
eligible = joined_dfs["target_eligible"]
joined_dfs.loc[eligible, TARGET_COLUMN] = (
    joined_dfs.loc[eligible]
    .groupby(["save", "current_owner"])["development_clicks"]
    .rank(method="average", pct=True)
)

joined_dfs["high_investment_target"] = pd.NA
joined_dfs.loc[eligible, "high_investment_target"] = (
    joined_dfs.loc[eligible, TARGET_COLUMN] >= 0.75
).astype(int)

LEAKAGE_COLUMNS = [
    "development_clicks", "new_tax", "new_production", "new_manpower",
    "changed_tax", "changed_production", "changed_manpower",
    "num_buildings", "changed_owner_num", "changed_controller_num",
    "legacy_heuristic_score", TARGET_COLUMN, "high_investment_target",
]
training_df = joined_dfs.loc[eligible].copy()

In [12]:
joined_dfs.loc[joined_dfs["province_id"] == 596]

,province_id,new_tax,new_production,new_manpower,changed_owner_num,changed_controller_num,num_buildings,current_owner,development_clicks,is_player_owned,...,middle_x_pos,middle_y_pos,changed_tax,changed_production,changed_manpower,legacy_heuristic_score,has_source_data,target_eligible,investment_preference_target,high_investment_target
595,596,7,7,3,0,0,0,NaN,0,False,...,4407.0,1362.0,0,0,0,0,True,False,NaN,<NA>
5536,596,5,7,2,4,8,3,GBR,0,True,...,4407.0,1362.0,-2,0,-1,4,True,True,0.319742,0
10477,596,7,7,3,2,1,3,MLC,0,False,...,4407.0,1362.0,0,0,0,5,True,False,NaN,<NA>
15418,596,7,7,3,0,0,0,NaN,0,False,...,4407.0,1362.0,0,0,0,0,True,False,NaN,<NA>
20359,596,8,7,3,2,1,1,MLC,0,False,...,4407.0,1362.0,1,0,0,4,True,False,NaN,<NA>
25300,596,9,7,3,2,1,3,MLC,0,False,...,4407.0,1362.0,2,0,0,7,True,False,NaN,<NA>
30241,596,6,7,2,2,3,4,MLC,1,False,...,4407.0,1362.0,-1,0,-1,4,True,False,NaN,<NA>
35182,596,8,7,3,3,19,4,GBR,0,True,...,4407.0,1362.0,1,0,0,8,True,True,0.341102,0
40123,596,7,7,3,3,11,4,BEI,0,False,...,4407.0,1362.0,0,0,0,7,True,False,NaN,<NA>
45064,596,7,7,3,2,1,3,MLC,0,False,...,4407.0,1362.0,0,0,0,5,True,False,NaN,<NA>


In [13]:
joined_dfs.to_csv(r'./data/provinces_stage3.csv')
joined_dfs

,province_id,new_tax,new_production,new_manpower,changed_owner_num,changed_controller_num,num_buildings,current_owner,development_clicks,is_player_owned,...,middle_x_pos,middle_y_pos,changed_tax,changed_production,changed_manpower,legacy_heuristic_score,has_source_data,target_eligible,investment_preference_target,high_investment_target
0,1,8,14,5,2,5,5,SWE,11,True,...,3077.0,321.0,3,9,2,21,True,True,0.9375,1
1,2,4,7,2,2,1,2,SWE,7,True,...,3050.0,349.0,2,5,0,11,True,True,0.8000,1
2,3,2,4,1,2,1,2,SWE,2,True,...,3050.0,376.0,0,2,0,6,True,True,0.4375,0
3,4,2,8,5,2,7,3,SWE,8,True,...,3045.0,309.0,0,5,3,13,True,True,0.8250,1
4,5,2,7,4,2,1,3,SWE,10,True,...,3011.0,318.0,1,6,3,15,True,True,0.9000,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
69169,4937,1,1,1,2,1,0,LAI,0,False,...,5584.0,1677.0,0,0,0,2,True,False,NaN,<NA>
69170,4938,2,2,1,2,1,0,VNL,0,False,...,5571.0,1664.0,0,0,0,2,True,False,NaN,<NA>
69171,4939,3,4,1,2,1,0,MAA,0,False,...,5591.0,1877.0,0,0,0,2,True,False,NaN,<NA>
69172,4940,0,0,0,0,0,0,NaN,0,False,...,938.0,580.0,0,0,0,0,True,False,NaN,<NA>
